# Laya on CPU vs GPU

[Laya](https://huggingface.co/convaiinnovations/laya) is a _decision model_: instead of generating text, it answers a fixed set of
questions about some input in **one forward pass** and returns calibrated probabilities.

This notebook walks through:

1. Asking Laya questions about a single message (the three question types)
2. Reading the result
3. Running the same model on **CPU** and on **GPU**
4. Timing both — one request at a time (latency) and many at once (throughput)

> The first run downloads the `english` checkpoint from Hugging Face (cached afterwards in `~/.cache/huggingface`).


## 1. Setup — what hardware do we have?


In [1]:
import json
import time

import torch
from laya import Router

print("torch       :", torch.__version__)
print("CUDA usable :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU         :", torch.cuda.get_device_name(0))
print("CPU threads :", torch.get_num_threads())

torch       : 2.14.1+cu130
CUDA usable : True
GPU         : NVIDIA GeForce RTX 4070 Ti SUPER
CPU threads : 20


## 2. Questions

Every request is a **state** (the text to judge) plus a dict of **questions**. Laya supports three question types:

| type     | answers                              | you get back                             |
| -------- | ------------------------------------ | ---------------------------------------- |
| `choice` | pick one of several labelled options | the chosen key + probability per option  |
| `score`  | place the input on an ordered scale  | an expected score between the scale ends |
| `noul`   | a yes/no question                    | probability that the answer is _yes_     |

`criteria` describes each option in plain words — that text is what the model reads, so make it descriptive.


In [2]:
questions = {
    "department": {
        "type": "choice",
        "instructions": "Which department should handle this?",
        "criteria": {
            "billing": "invoices, payments, refunds",
            "technical": "bugs, outages, system errors",
            "other": "everything else",
        },
    },
    "urgency": {
        "type": "score",
        "instructions": "How urgent is this?",
        "criteria": ["not urgent", "soon", "blocking"],
    },
    "churn_risk": {
        "type": "noul",
        "instructions": "Does the user threaten to cancel or leave?",
    },
}

state = "Hi, we were billed twice for March. Please refund the duplicate today or we will cancel our plan."

## 3. One prediction on the CPU

`Router` picks the right checkpoint for the input (`english`, `multilingual`, or `typed-decisions`) and loads it lazily.
`device=` decides where the model lives. We use `cpu` first.


In [3]:
t0 = time.perf_counter()
cpu_router = Router(device="cpu")
cpu_router.preload(["english"])  # load now, so the timings below don't include loading
print(f"CPU model load: {time.perf_counter() - t0:.1f}s")

result = cpu_router.predict(state, questions)
print(json.dumps(result, indent=2, default=str))

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

/mnt/jinho/Development/Projects/2026/TuringCollege-AE/AE/sprint1/laya/.venv/lib/python3.12/site-packages/laya/router.py:774: RuntimeWarning: laya: this checkpoint ships invalid temperatures or values outside [0.5, 5]; using choice:11+=0.10058280825614929 -> 0.5. Treat confidence from the affected entries as uncalibrated.
  return Agent(repo, **kwargs)


CPU model load: 295.7s
{
  "model": "laya-rl-agent",
  "answers": {
    "department": {
      "type": "choice",
      "choice": "billing",
      "probabilities": {
        "billing": 0.9865,
        "technical": 0.008,
        "other": 0.0055
      },
      "confidence": 0.9267,
      "answer_confidence": 0.9865,
      "action": {
        "act_probability": 1.0
      }
    },
    "urgency": {
      "type": "score",
      "score": 1.7722,
      "legend": {
        "0": "not urgent",
        "1": "soon",
        "2": "blocking"
      },
      "probabilities": {
        "0": 0.0389,
        "1": 0.15,
        "2": 0.8111
      },
      "confidence": 0.4714,
      "answer_confidence": 0.8111,
      "action": {
        "act_probability": 1.0
      }
    },
    "churn_risk": {
      "type": "noul",
      "noul": 0.879,
      "confidence": 0.879,
      "answer_confidence": 0.879,
      "action": {
        "act_probability": 1.0
      }
    }
  },
  "usage": {
    "input_tokens": 164,
    "out

### Reading the result

Everything lives under `result["answers"][<question id>]`; `result["routing"]` says which checkpoint answered and why.


In [4]:
a = result["answers"]
print("department :", a["department"]["choice"], f"(confidence {a['department']['confidence']:.2f})")
print("urgency    :", a["urgency"]["score"])
print("churn risk :", f"{a['churn_risk']['noul']:.2%} probability of 'yes'")
print("model used :", result["routing"]["model"])

department : billing (confidence 0.93)
urgency    : 1.7722
churn risk : 87.90% probability of 'yes'
model used : english


## 4. The same thing on the GPU

Only the `device` argument changes. Laya moves the model to the GPU and uses mixed precision automatically.

> **Why was the CPU load so slow?** The first `Router` pays for importing torch/transformers and reading the weights from disk. The GPU router reuses all of that, so its "load" time is only the transfer to the GPU. Compare _inference_ times (sections 7–8), not load times.


In [5]:
assert torch.cuda.is_available(), "No CUDA GPU visible - the GPU sections below will not work."

t0 = time.perf_counter()
gpu_router = Router(device="cuda")
gpu_router.preload(["english"])
print(f"GPU model load: {time.perf_counter() - t0:.1f}s")

gpu_result = gpu_router.predict(state, questions)
print("department :", gpu_result["answers"]["department"]["choice"])
print("churn risk :", f"{gpu_result['answers']['churn_risk']['noul']:.2%}")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

GPU model load: 1.2s
department : billing
churn risk : 88.17%


## 5. A small test set

To time things fairly we need more than one message. Here are 32 support tickets (8 templates × 4 copies).


In [6]:
tickets = [
    "Hi, we were billed twice for March. Please refund the duplicate today or we will cancel our plan.",
    "The dashboard has been down for two hours and none of our team can log in.",
    "Could you send me last year's invoices as PDFs? No rush.",
    "Your app crashes every time I upload a photo larger than 5 MB.",
    "I love the new dark mode, thanks!",
    "If this outage isn't fixed by tonight we're moving to a competitor.",
    "How do I change the email address on my account?",
    "My card was charged but the order shows as unpaid. Please sort this out.",
] * 4
len(tickets)

32

## 6. Timing helper

Two things make GPU timings tricky:

- **Warm-up** — the first call pays one-off costs (CUDA kernels, memory allocation). We run a few untimed calls first.
- **Asynchrony** — GPU work is queued; `torch.cuda.synchronize()` waits until it really finishes before we stop the clock.


In [7]:
def timeit(fn, device, repeats=5, warmup=2):
    # Return the median wall-clock seconds of fn() over `repeats` runs.
    for _ in range(warmup):
        fn()
    times = []
    for _ in range(repeats):
        if device == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        fn()
        if device == "cuda":
            torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
    return sorted(times)[len(times) // 2]

## 7. Latency — one request at a time

This is what an interactive app sees: a single message comes in, we want the answer back.


In [8]:
latency = {
    "cpu": timeit(lambda: cpu_router.predict(state, questions), "cpu"),
    "cuda": timeit(lambda: gpu_router.predict(state, questions), "cuda"),
}
for dev, s in latency.items():
    print(f"{dev:>4}: {s * 1000:7.1f} ms per request")
print(f"GPU speed-up: {latency['cpu'] / latency['cuda']:.1f}x")

 cpu:   187.3 ms per request
cuda:    12.3 ms per request
GPU speed-up: 15.2x


## 8. Throughput — many requests in one batch

`predict_batch` packs many messages into shared forward passes. This is where a GPU really pays off: it can process
the whole batch in parallel, while a CPU mostly works through it row by row.

Each request is a dict with `state` and `questions`; results come back in the same order.


In [12]:
requests = [{"state": t, "questions": questions} for t in tickets]


def run_batch(router, n):
    return router.predict_batch(requests[:n])


rows = []
for n in [1, 8, 32]:
    cpu_s = timeit(lambda: run_batch(cpu_router, n), "cpu", repeats=3, warmup=1)
    gpu_s = timeit(lambda: run_batch(gpu_router, n), "cuda", repeats=3, warmup=1)
    rows.append((n, cpu_s, gpu_s))

print(f"{'batch':>5} | {'CPU total':>10} | {'GPU total':>10} | {'CPU / msg':>10} | {'GPU / msg':>10} | speed-up")
print("-" * 72)
for n, c, g in rows:
    print(
        f"{n:>5} | {c * 1000:>8.1f}ms | {g * 1000:>8.1f}ms | {c / n * 1000:>8.1f}ms | {g / n * 1000:>8.1f}ms | {c / g:>6.1f}x"
    )

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

/mnt/jinho/Development/Projects/2026/TuringCollege-AE/AE/sprint1/laya/.venv/lib/python3.12/site-packages/laya/router.py:774: RuntimeWarning: laya: this checkpoint ships invalid temperatures or values outside [0.5, 5]; using choice:11+=0.10058280825614929 -> 0.5. Treat confidence from the affected entries as uncalibrated.
  return Agent(repo, **kwargs)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

batch |  CPU total |  GPU total |  CPU / msg |  GPU / msg | speed-up
------------------------------------------------------------------------
    1 |    193.8ms |     11.6ms |    193.8ms |     11.6ms |   16.8x
    8 |   1219.5ms |     32.4ms |    152.4ms |      4.1ms |   37.6x
   32 |   4882.0ms |    114.9ms |    152.6ms |      3.6ms |   42.5x


## 9. Do CPU and GPU agree?

The GPU runs in reduced precision (bf16), so probabilities can differ slightly, but the decisions should match.


In [10]:
cpu_out = cpu_router.predict_batch(requests[:8])
gpu_out = gpu_router.predict_batch(requests[:8])

print(f"{'ticket':<60} {'CPU':>9} {'GPU':>9} {'churn CPU':>10} {'churn GPU':>10}")
for t, c, g in zip(tickets[:8], cpu_out, gpu_out):
    ca, ga = c["answers"], g["answers"]
    print(
        f"{t[:58]:<60} {ca['department']['choice']:>9} {ga['department']['choice']:>9} "
        f"{ca['churn_risk']['noul']:>10.2f} {ga['churn_risk']['noul']:>10.2f}"
    )

ticket                                                             CPU       GPU  churn CPU  churn GPU
Hi, we were billed twice for March. Please refund the dupl     billing   billing       0.88       0.88
The dashboard has been down for two hours and none of our    technical technical       0.08       0.08
Could you send me last year's invoices as PDFs? No rush.       billing   billing       0.00       0.00
Your app crashes every time I upload a photo larger than 5   technical technical       0.31       0.31
I love the new dark mode, thanks!                                other     other       0.00       0.00
If this outage isn't fixed by tonight we're moving to a co   technical technical       0.79       0.80
How do I change the email address on my account?             technical technical       0.00       0.00
My card was charged but the order shows as unpaid. Please      billing   billing       0.11       0.11


## 10. Clean up

Free the GPU memory when you're done (or just restart the kernel).


In [11]:
cpu_router.unload()
gpu_router.unload()
torch.cuda.empty_cache()

## Takeaways

- Switching hardware is one argument: `Router(device="cpu")` vs `Router(device="cuda")` (omit it to auto-pick the GPU).
- Even for **one request** the GPU is much faster (~200 ms on CPU vs ~12 ms on GPU on an RTX 4070 Ti SUPER).
- With **batches** the gap grows: the CPU stays at ~170 ms per message, while the GPU drops to ~3 ms (about 50x faster). Use `predict_batch` whenever you have many inputs.
- Always warm up and `torch.cuda.synchronize()` before timing GPU code, otherwise the numbers lie.
